# Customer Growth Intelligence — Online Retail II branch

**Run on Kaggle:** *Add Input* → search **"Online Retail II UCI"** (a CSV or the original Excel file both work) → **Save Version → Save & Run All**.

Why this dataset: real transactions of a UK online gift/homeware retailer, Dec 2009 – Dec 2011, with **customers who buy
repeatedly** (what Olist lacks) and **cancellations/returns** (an "order went wrong" signal) for the *same* customers.

| Olist model | This branch | Question |
|---|---|---|
| v2 retention | **R1** | Will this customer buy again in the next 90 days? |
| v3 order risk | **O1** | Will (part of) this order be cancelled/returned within 30 days? |
| link | **L1** | Do customers with a recent return come back less? |

**Success criteria, fixed before running:**
* **R1** succeeds if, on later months it never saw, its AUC interval's lower end is **≥ 0.65** *and* above the best
  single-number rule (recency, frequency or spend). Beating those simple rules is the real test.
* **O1** succeeds if its AUC interval's lower end is above the best single-feature rule.

Caveat: in this business a "return" is often a customer adjusting a wholesale order, not a service failure.

In [ ]:
import os, math, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
try:
    display
except NameError:
    display = print
SEED = 42
H_RET, H_O = 90, 30                 # retention horizon, return window (days)
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs"); OUT_DIR.mkdir(parents=True, exist_ok=True)

def find_files():
    roots = [os.environ["RETAIL_DIR"]] if os.environ.get("RETAIL_DIR") else []
    roots += ["/kaggle/input", "."]
    found = []
    for root in roots:
        for dp, _, files in os.walk(root):
            for f in files:
                if "retail" in f.lower() and f.lower().endswith((".csv", ".xlsx", ".xls")):
                    found.append(Path(dp) / f)
        if found:
            return found
    raise FileNotFoundError("Attach the Online Retail II dataset as an input (or set RETAIL_DIR).")

COLS = {"invoice": ["Invoice", "InvoiceNo"], "stock_code": ["StockCode"], "description": ["Description"], "quantity": ["Quantity"],
        "invoice_date": ["InvoiceDate"], "price": ["Price", "UnitPrice"], "customer_id": ["Customer ID", "CustomerID"], "country": ["Country"]}

def read_any(p):
    if p.suffix.lower() == ".csv":
        for enc in ("utf-8", "latin-1"):
            try:
                return [pd.read_csv(p, encoding=enc, dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str})]
            except UnicodeDecodeError:
                continue
    return list(pd.read_excel(p, sheet_name=None, dtype={"Invoice": str, "InvoiceNo": str, "StockCode": str}).values())

frames = []
for p in find_files():
    for d in read_any(p):
        ren = {}
        for std, opts in COLS.items():
            hit = [c for c in opts if c in d.columns]
            if hit:
                ren[hit[0]] = std
        if len(ren) == len(COLS):
            frames.append(d.rename(columns=ren)[list(COLS)])
        print(f"{p.name}: {len(d):,} rows {'(used)' if len(ren) == len(COLS) else '(skipped: columns not recognised)'}")
raw = pd.concat(frames, ignore_index=True)
raw["invoice"] = raw.invoice.astype(str).str.strip(); raw["stock_code"] = raw.stock_code.astype(str).str.strip().str.upper()
raw["invoice_date"] = pd.to_datetime(raw.invoice_date)
n0 = len(raw); raw = raw.drop_duplicates(); print(f"Rows {n0:,} -> {len(raw):,} after removing exact duplicates (also removes overlap if two file versions were attached)")

In [ ]:
NON_PRODUCT = {"POST", "D", "M", "C2", "DOT", "BANK CHARGES", "AMAZONFEE", "CRUK", "S", "PADS", "B", "ADJUST", "ADJUST2", "TEST001", "TEST002", "SP1002"}
is_cancel = raw.invoice.str.startswith("C") | (raw.quantity < 0)
prod = ~raw.stock_code.isin(NON_PRODUCT) & raw.stock_code.str.match(r"^\d")
df = raw[raw.customer_id.notna() & prod & (raw.price > 0)].copy()
df["customer_id"] = df.customer_id.astype(float).astype(int).astype(str)
df["is_return"] = is_cancel.loc[df.index]
df["line_value"] = df.quantity.abs() * df.price
END = df.invoice_date.max()
pl = df[~df.is_return & (df.quantity > 0)].copy()          # purchase lines
rl = df[df.is_return].copy()                               # cancellation/return lines

inv = pl.groupby("invoice").agg(customer_id=("customer_id", lambda s: s.mode().iat[0]), date=("invoice_date", "min"),
                                n_lines=("stock_code", "size"), n_products=("stock_code", "nunique"), units=("quantity", "sum"),
                                value=("line_value", "sum"), max_qty=("quantity", "max"), country=("country", "first")).reset_index()
print(f"Data: {df.invoice_date.min().date()} to {END.date()} | customers {df.customer_id.nunique():,} | purchase invoices {len(inv):,} | "
      f"return lines {len(rl):,} ({len(rl)/len(df)*100:.1f}% of lines)")
per_c = inv.groupby("customer_id").size()
print(f"Customers with 2+ purchase invoices: {(per_c >= 2).mean()*100:.1f}% (Olist: about 3%) | median invoices per customer: {per_c.median():.0f}")
TOP_COUNTRIES = inv.country.value_counts().head(8).index.tolist()

def wilson(k, n, z=1.959964):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (100 * max(0, c - h), 100 * min(1, c + h))

## R1. Retention: will a customer buy again in the next 90 days?
Monthly snapshots. At each snapshot date T, every customer who bought in the previous 365 days is described **only with
history before T**; the outcome is any purchase in (T, T+90]. Training snapshots end their 90-day window before the first
test snapshot, so nothing from the test period leaks in. The last snapshots are the test.

In [ ]:
R_NUM_BASE = ["recency_days", "tenure_days", "freq_all", "freq_90", "freq_180", "freq_365", "log_value_365", "log_aov", "value_trend",
              "gap_mean", "gap_cv", "overdue_ratio", "log_products_365", "log_units_per_line", "returns_365", "return_share_365", "is_uk"]

def customer_features(T):
    h = inv[inv.date < T]; r_ = rl[rl.invoice_date < T]
    g = h.groupby("customer_id")
    f = pd.DataFrame({"last": g.date.max(), "first": g.date.min(), "freq_all": g.size(), "aov": g.value.mean(),
                      "units_per_line": g.units.sum() / g.n_lines.sum(), "country": g.country.agg(lambda s: s.mode().iat[0])})
    f = f[f["last"] >= T - pd.Timedelta(days=365)]
    for d in (90, 180, 365):
        w = h[h.date >= T - pd.Timedelta(days=d)].groupby("customer_id")
        f[f"freq_{d}"] = w.size(); f[f"value_{d}"] = w.value.sum()
    prev = h[(h.date >= T - pd.Timedelta(days=180)) & (h.date < T - pd.Timedelta(days=90))].groupby("customer_id").value.sum()
    f = f.join(prev.rename("value_prev90")).fillna({c: 0 for c in ["freq_90", "freq_180", "freq_365", "value_90", "value_180", "value_365", "value_prev90"]})
    # gaps between purchase DAYS: several invoices on the same day count as one occasion (otherwise gaps of 0 -> division by zero)
    days_ = h.assign(day=h.date.dt.normalize()).drop_duplicates(["customer_id", "day"]).sort_values("day")
    gaps = days_.groupby("customer_id").day.apply(lambda s: s.diff().dt.total_seconds().div(86400).dropna())
    gs = gaps.groupby(level=0).agg(["mean", "std"]) if len(gaps) else pd.DataFrame(columns=["mean", "std"])
    f = f.join(gs.rename(columns={"mean": "gap_mean", "std": "gap_std"}))
    np365 = pl[(pl.invoice_date < T) & (pl.invoice_date >= T - pd.Timedelta(days=365))].groupby("customer_id").stock_code.nunique()
    rr = r_[r_.invoice_date >= T - pd.Timedelta(days=365)].groupby("customer_id").agg(returns_365=("invoice", "nunique"), return_value_365=("line_value", "sum"))
    f = f.join(np365.rename("products_365")).join(rr).fillna({"returns_365": 0, "return_value_365": 0, "products_365": 0})
    out = pd.DataFrame(index=f.index)
    out["recency_days"] = (T - f["last"]).dt.total_seconds() / 86400
    out["tenure_days"] = (T - f["first"]).dt.total_seconds() / 86400
    for c in ["freq_all", "freq_90", "freq_180", "freq_365"]:
        out[c] = f[c]
    out["log_value_365"] = np.log1p(f.value_365); out["log_aov"] = np.log1p(f.aov)
    out["value_trend"] = np.log1p(f.value_90) - np.log1p(f.value_prev90)
    out["gap_mean"] = f.gap_mean; out["gap_cv"] = f.gap_std / f.gap_mean
    out["overdue_ratio"] = out.recency_days / f.gap_mean
    out["log_products_365"] = np.log1p(f.products_365); out["log_units_per_line"] = np.log1p(f.units_per_line)
    out["returns_365"] = f.returns_365; out["return_share_365"] = f.return_value_365 / (f.value_365 + f.return_value_365 + 1e-9)
    out["is_uk"] = (f.country == "United Kingdom").astype(float)
    out["country_group"] = f.country.where(f.country.isin(TOP_COUNTRIES), "Other")
    out["snapshot_month"] = float(T.month)
    out[R_NUM_BASE] = out[R_NUM_BASE].replace([np.inf, -np.inf], np.nan)      # safety net: no infinite values reach the model
    nxt = inv[(inv.date > T) & (inv.date <= T + pd.Timedelta(days=H_RET))].customer_id.unique()
    out["bought_next_90d"] = out.index.isin(nxt).astype(int)
    out["snapshot"] = T
    return out.reset_index().rename(columns={"index": "customer_id"})

first_snap = (inv.date.min() + pd.Timedelta(days=180)).to_period("M").to_timestamp() + pd.offsets.MonthBegin(1)
snaps = list(pd.date_range(first_snap, END - pd.Timedelta(days=H_RET), freq="MS"))
n_test = min(4, max(1, len(snaps) // 3))
TEST_SNAPS = snaps[-n_test:]
TRAIN_SNAPS = [s for s in snaps if s + pd.Timedelta(days=H_RET) <= TEST_SNAPS[0]]
panel = pd.concat([customer_features(T) for T in snaps], ignore_index=True)
R_NUM = ["recency_days", "tenure_days", "freq_all", "freq_90", "freq_180", "freq_365", "log_value_365", "log_aov", "value_trend",
         "gap_mean", "gap_cv", "overdue_ratio", "log_products_365", "log_units_per_line", "returns_365", "return_share_365", "is_uk", "snapshot_month"]
R_CAT = ["country_group"]
R_FEATS = R_NUM + R_CAT
print(f"Snapshots: {len(snaps)} ({snaps[0].date()} .. {snaps[-1].date()}); train {len(TRAIN_SNAPS)}, test {[s.date().isoformat() for s in TEST_SNAPS]}")
s = panel.groupby("snapshot").agg(customers=("customer_id", "size"), bought_next_90d_pct=("bought_next_90d", lambda x: round(x.mean() * 100, 1)))
display(s.T)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import joblib, sklearn

def make_logit(num, cat):
    parts = [("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=False)), ("sc", StandardScaler()),
                               ("clip", FunctionTransformer(np.clip, kw_args={"a_min": -5, "a_max": 5}))]), num)]
    if cat: parts.append(("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30), cat))
    return Pipeline([("pre", ColumnTransformer(parts)), ("clf", LogisticRegression(max_iter=3000, C=0.5))])

def make_hgb(num, cat):
    pre = ColumnTransformer([("num", "passthrough", num)] + ([("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat)] if cat else []))
    return Pipeline([("pre", pre), ("clf", HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, min_samples_leaf=50,
                                                                          l2_regularization=1.0, categorical_features=list(range(len(num), len(num) + len(cat))) or None,
                                                                          random_state=SEED))])

def cluster_boot_auc(y, p, groups, n=300):
    """Bootstrap over customers (a customer appears in several snapshots/orders, so rows aren't independent)."""
    r = np.random.default_rng(SEED); codes, uniq = pd.factorize(groups)
    idx_by = pd.Series(np.arange(len(y))).groupby(codes).apply(np.array).values
    out = []
    for _ in range(n):
        rows = np.concatenate(idx_by[r.integers(0, len(idx_by), len(idx_by))])
        if y[rows].min() != y[rows].max():
            out.append(roc_auc_score(y[rows], p[rows]))
    return [float(v) for v in np.percentile(out, [2.5, 97.5])]

def band_table(p_train, p, y):
    q50, q90 = np.quantile(p_train, [.5, .9]); b = np.where(p >= q90, "High", np.where(p >= q50, "Medium", "Low"))
    rows = []
    for name in ("High", "Medium", "Low"):
        yy = y[b == name]; k, n = int(yy.sum()), len(yy); lo, hi = wilson(k, n)
        rows.append(dict(band=name, n=n, rate_pct=round(100 * k / max(n, 1), 2), ci_low=round(lo, 2), ci_high=round(hi, 2)))
    return rows

tr = panel[panel.snapshot.isin(TRAIN_SNAPS)]; te = panel[panel.snapshot.isin(TEST_SNAPS)]
y_tr, y_te = tr.bought_next_90d.values, te.bought_next_90d.values
R_RES = {}
for name, f in [("logistic", make_logit), ("gradient boosting", make_hgb)]:
    pipe = f(R_NUM, R_CAT).fit(tr[R_FEATS], y_tr); p = pipe.predict_proba(te[R_FEATS])[:, 1]
    R_RES[name] = dict(pipe=pipe, p=p, p_tr=pipe.predict_proba(tr[R_FEATS])[:, 1], auc=roc_auc_score(y_te, p), ci=cluster_boot_auc(y_te, p, te.customer_id.values))
base_rules = {}
for f_, sign in [("recency_days", -1), ("freq_365", 1), ("log_value_365", 1), ("freq_90", 1), ("overdue_ratio", -1)]:
    x = te[f_].fillna(te[f_].median()).values * sign
    base_rules[f_] = dict(auc=roc_auc_score(y_te, x), ci=cluster_boot_auc(y_te, x, te.customer_id.values))
first_only = te.freq_all.values == 1

print(f"Train: {len(tr):,} customer-snapshots ({y_tr.mean()*100:.1f}% bought again) | Test: {len(te):,} ({y_te.mean()*100:.1f}%) on {len(TEST_SNAPS)} later months")
rows = [dict(model=k, auc=round(v["auc"], 3), ci=f"{v['ci'][0]:.3f}-{v['ci'][1]:.3f}") for k, v in R_RES.items()]
rows += [dict(model=f"rule: {k}", auc=round(v["auc"], 3), ci=f"{v['ci'][0]:.3f}-{v['ci'][1]:.3f}") for k, v in base_rules.items()]
display(pd.DataFrame(rows))
R_BEST = max(R_RES, key=lambda k: R_RES[k]["auc"]); rb = R_RES[R_BEST]
best_rule = max(base_rules, key=lambda k: base_rules[k]["auc"])
print(f"Customers with only ONE earlier purchase (the Olist situation): AUC {roc_auc_score(y_te[first_only], rb['p'][first_only]):.3f} "
      f"on {first_only.sum():,} rows ({y_te[first_only].mean()*100:.1f}% bought again)" if first_only.sum() > 50 and 0 < y_te[first_only].mean() < 1 else "")
print(f"{R_BEST} by band on the test months:"); display(pd.DataFrame(band_table(rb["p_tr"], rb["p"], y_te)))
R_PASS = rb["ci"][0] >= 0.65 and rb["ci"][0] > base_rules[best_rule]["auc"]
print(f"R1 criterion: CI low {rb['ci'][0]:.3f} >= 0.65 and > best rule ({best_rule}, AUC {base_rules[best_rule]['auc']:.3f}) -> {'PASS' if R_PASS else 'FAIL'}")

## O1. Order risk: will (part of) an order be cancelled/returned within 30 days?
Each cancellation/return line is matched to the customer's most recent earlier purchase of the same product. Features
are known when the order is placed, including the customer's and the products' **return history up to that moment**
(point-in-time, checked below). Test = the last ~5 months with a resolved 30-day window; training ends 30 days earlier.

In [ ]:
p_sorted = pl.sort_values("invoice_date")[["invoice", "customer_id", "stock_code", "invoice_date"]].rename(columns={"invoice_date": "p_date"})
r_sorted = rl.sort_values("invoice_date")[["customer_id", "stock_code", "invoice_date"]].rename(columns={"invoice_date": "r_date"})
m = pd.merge_asof(r_sorted, p_sorted, left_on="r_date", right_on="p_date", by=["customer_id", "stock_code"], direction="backward")
m = m[m.invoice.notna() & ((m.r_date - m.p_date).dt.total_seconds() / 86400 <= H_O)]
returned_inv = set(m.invoice)
inv["returned_30d"] = inv.invoice.isin(returned_inv).astype(int)
print(f"Return lines matched to an earlier purchase within {H_O} days: {len(m):,} of {len(rl):,}; "
      f"orders with a return: {inv.returned_30d.mean()*100:.2f}%")

def prior_rate(events_key, ev_time, ev_val, q_key, q_time, a, prior):
    """Point-in-time shrunk rate: events known strictly before each query time."""
    e = pd.DataFrame({"k": events_key.values, "t": pd.to_datetime(ev_time.values).astype("datetime64[ns]"), "v": ev_val.values}).sort_values("t")
    e["n"] = e.groupby("k").cumcount() + 1; e["s"] = e.groupby("k").v.cumsum()
    q = pd.DataFrame({"k": q_key.values, "t": pd.to_datetime(q_time.values).astype("datetime64[ns]"), "i": np.arange(len(q_key))}).sort_values("t")
    j = pd.merge_asof(q, e[["k", "t", "n", "s"]], on="t", by="k", allow_exact_matches=False, direction="backward").sort_values("i")
    n, s = j.n.fillna(0).values, j.s.fillna(0).values
    return (s + a * prior) / (n + a), n

# Each purchase line's return becomes KNOWN at the return date (or after the 30-day window if not returned)
line = pl[["invoice", "customer_id", "stock_code", "invoice_date"]].copy()
mm = m[["invoice", "stock_code", "r_date"]].drop_duplicates(["invoice", "stock_code"])
line = line.merge(mm, on=["invoice", "stock_code"], how="left")
line["ret"] = line.r_date.notna().astype(int)
line["known_at"] = line.r_date.fillna(line.invoice_date + pd.Timedelta(days=H_O))
O_TEST_START = (END - pd.Timedelta(days=H_O) - pd.Timedelta(days=150)).normalize()
O_TRAIN_END = O_TEST_START - pd.Timedelta(days=H_O)
prior_line = line.loc[line.known_at < O_TRAIN_END, "ret"].mean()
prior_inv = inv.loc[inv.date + pd.Timedelta(days=H_O) < O_TRAIN_END, "returned_30d"].mean()

line["prod_rate"], _ = prior_rate(line.stock_code, line.known_at, line.ret, line.stock_code, line.invoice_date, 30, prior_line)
pr = line.groupby("invoice").prod_rate.agg(prod_rate_max="max", prod_rate_mean="mean").reset_index()
inv_known = inv.date + pd.Timedelta(days=H_O)
inv_r = inv[["invoice", "customer_id", "date"]].assign(known=inv.returned_30d.values)
ret_known = inv.date.copy()
ret_known.loc[inv.returned_30d == 1] = inv.loc[inv.returned_30d == 1, "invoice"].map(m.groupby("invoice").r_date.min())
ret_known = ret_known.where(inv.returned_30d == 1, inv_known)
inv["cust_return_rate"], inv["cust_prior_orders_resolved"] = prior_rate(inv.customer_id, ret_known, inv.returned_30d, inv.customer_id, inv.date, 5, prior_inv)
inv = inv.sort_values(["date", "invoice"]).reset_index(drop=True)
inv["cust_prior_orders"] = inv.groupby("customer_id").cumcount()
inv["days_since_prev"] = inv.groupby("customer_id").date.diff().dt.total_seconds() / 86400
inv = inv.merge(pr, on="invoice", how="left", validate="one_to_one")

# leakage check: brute-force the customer rate for 200 random orders
rng = np.random.default_rng(SEED); bad = 0
kn = pd.DataFrame({"customer_id": inv.customer_id, "t": inv.invoice.map(dict(zip(inv_r.invoice, ret_known.values))), "v": inv.returned_30d})
for i in rng.choice(len(inv), 200, replace=False):
    rr = inv.iloc[i]; e = kn[(kn.customer_id == rr.customer_id) & (kn.t < rr.date)]
    if abs((e.v.sum() + 5 * prior_inv) / (len(e) + 5) - rr.cust_return_rate) > 1e-9:
        bad += 1
assert bad == 0, f"{bad} orders see return information from the future"
print("Leakage check passed: customer return history is point-in-time for 200/200 random orders.")

inv["log_value"] = np.log1p(inv.value); inv["log_units"] = np.log1p(inv.units); inv["log_max_qty"] = np.log1p(inv.max_qty)
inv["avg_unit_price_log"] = np.log1p(inv.value / inv.units); inv["log_prior_orders"] = np.log1p(inv.cust_prior_orders)
inv["is_first_order"] = (inv.cust_prior_orders == 0).astype(float)
inv["hour"] = inv.date.dt.hour.astype(float); inv["weekday"] = inv.date.dt.dayofweek.astype(float); inv["month"] = inv.date.dt.month.astype(float)
inv["country_group"] = inv.country.where(inv.country.isin(TOP_COUNTRIES), "Other")
O_NUM = ["n_lines", "n_products", "log_value", "log_units", "log_max_qty", "avg_unit_price_log", "log_prior_orders", "is_first_order",
         "days_since_prev", "cust_return_rate", "prod_rate_max", "prod_rate_mean", "hour", "weekday", "month"]
O_FEATS = O_NUM + ["country_group"]
inv[O_NUM] = inv[O_NUM].replace([np.inf, -np.inf], np.nan)

resolved = inv[inv.date <= END - pd.Timedelta(days=H_O)]
otr = resolved[resolved.date < O_TRAIN_END - pd.Timedelta(days=H_O)]   # outcome windows closed before training cut-off
ote = resolved[resolved.date >= O_TEST_START]
O_RES = {}
for name, f in [("logistic", make_logit), ("gradient boosting", make_hgb)]:
    pipe = f(O_NUM, ["country_group"]).fit(otr[O_FEATS], otr.returned_30d); p = pipe.predict_proba(ote[O_FEATS])[:, 1]
    O_RES[name] = dict(pipe=pipe, p=p, p_tr=pipe.predict_proba(otr[O_FEATS])[:, 1], auc=roc_auc_score(ote.returned_30d, p),
                       ci=cluster_boot_auc(ote.returned_30d.values, p, ote.customer_id.values))
o_rules = {f_: roc_auc_score(ote.returned_30d, ote[f_].fillna(ote[f_].median())) for f_ in ["cust_return_rate", "prod_rate_max", "n_lines", "log_value"]}
o_rules = {k: max(v, 1 - v) for k, v in o_rules.items()}
print(f"Train orders {len(otr):,} ({otr.returned_30d.mean()*100:.2f}% returned) | test orders {len(ote):,} ({ote.returned_30d.mean()*100:.2f}%) from {O_TEST_START.date()}")
display(pd.DataFrame([dict(model=k, auc=round(v["auc"], 3), ci=f"{v['ci'][0]:.3f}-{v['ci'][1]:.3f}") for k, v in O_RES.items()] +
                     [dict(model=f"rule: {k}", auc=round(v, 3), ci="") for k, v in o_rules.items()]))
O_BEST = max(O_RES, key=lambda k: O_RES[k]["auc"]); ob = O_RES[O_BEST]
print(f"{O_BEST} by band on the test period:"); display(pd.DataFrame(band_table(ob["p_tr"], ob["p"], ote.returned_30d.values)))
O_PASS = ob["ci"][0] > max(o_rules.values())
print(f"O1 criterion: CI low {ob['ci'][0]:.3f} > best rule ({max(o_rules, key=o_rules.get)}, {max(o_rules.values()):.3f}) -> {'PASS' if O_PASS else 'FAIL'}")

## L1. Link: do customers with a recent return buy again less? (test snapshots, observational)

In [ ]:
t = te.assign(recent_return=np.where(te.returns_365 > 0, "had a return (last 365d)", "no return"))
rows = []
for g_, d in t.groupby("recent_return"):
    k, n = int(d.bought_next_90d.sum()), len(d); lo, hi = wilson(k, n)
    rows.append(dict(group=g_, customers=n, bought_again_pct=round(100 * k / n, 1), ci_low=round(lo, 1), ci_high=round(hi, 1),
                     median_orders_365=d.freq_365.median()))
display(pd.DataFrame(rows))
print("Returners are usually MORE active customers here (more orders -> more chances to return), so compare at similar activity:")
t["activity"] = pd.qcut(t.freq_365.rank(method="first"), 4, labels=["lowest", "low", "high", "highest"])
display((t.groupby(["activity", "recent_return"], observed=True).bought_next_90d.mean() * 100).round(1).unstack())

## Export (always saved; the criteria above say how far to trust each model)

In [ ]:
VER = pd.Timestamp.now().strftime("%Y%m%d")
final_r = (make_hgb if R_BEST == "gradient boosting" else make_logit)(R_NUM, R_CAT).fit(panel[panel.snapshot.isin(TRAIN_SNAPS + TEST_SNAPS)][R_FEATS],
                                                                                         panel[panel.snapshot.isin(TRAIN_SNAPS + TEST_SNAPS)].bought_next_90d)
joblib.dump({"model_version": f"retail-retention-r1-{VER}", "kind": "retention", "dataset": "Online Retail II", "pipeline": final_r,
             "features": R_FEATS, "num": R_NUM, "cat": R_CAT, "horizon_days": H_RET, "sklearn_version": sklearn.__version__,
             "base_rate": float(y_te.mean()),
             "score_quantiles": {"p50": float(np.quantile(rb["p_tr"], .5)), "p90": float(np.quantile(rb["p_tr"], .9))},
             "metrics": {"model": R_BEST, "oot_auc": rb["auc"], "oot_auc_ci": rb["ci"], "best_rule": best_rule,
                         "best_rule_auc": base_rules[best_rule]["auc"], "passed": bool(R_PASS),
                         "bands_oot": band_table(rb["p_tr"], rb["p"], y_te)}}, OUT_DIR / "retail_retention_model.joblib")
final_o = (make_hgb if O_BEST == "gradient boosting" else make_logit)(O_NUM, ["country_group"]).fit(resolved[O_FEATS], resolved.returned_30d)
joblib.dump({"model_version": f"retail-order-risk-o1-{VER}", "kind": "order_risk", "dataset": "Online Retail II", "pipeline": final_o,
             "features": O_FEATS, "num": O_NUM, "cat": ["country_group"], "window_days": H_O, "sklearn_version": sklearn.__version__,
             "metrics": {"model": O_BEST, "oot_auc": ob["auc"], "oot_auc_ci": ob["ci"], "rules": o_rules, "passed": bool(O_PASS)}},
            OUT_DIR / "retail_order_risk_model.joblib")
# ---- extra exports for the app's dataset package -------------------------------------------------
# O1 bands (thresholds from the final model's scores) + their out-of-time outcomes
p_all_o = final_o.predict_proba(resolved[O_FEATS])[:, 1]
o_q = {"p50": float(np.quantile(p_all_o, .5)), "p90": float(np.quantile(p_all_o, .9))}
art_o = joblib.load(OUT_DIR / "retail_order_risk_model.joblib")
art_o.update(score_quantiles=o_q, base_rate=float(resolved.returned_30d.mean()), target="returned_30d",
             band_outcomes_oot=band_table(ob["p_tr"], ob["p"], ote.returned_30d.values))
joblib.dump(art_o, OUT_DIR / "retail_order_risk_model.joblib")
# every order with its purchase-time features and (if resolved) its outcome
orders_out = inv[["invoice", "customer_id", "date", "returned_30d"] + O_FEATS].copy()
orders_out["resolved"] = orders_out.date <= END - pd.Timedelta(days=H_O)
orders_out.loc[~orders_out.resolved, "returned_30d"] = np.nan
orders_out.to_csv(OUT_DIR / "retail_orders_scored.csv", index=False)
# the learned link: repeat buying with vs without a recent return, matched on activity (Mantel-Haenszel risk ratio)
lk = te.assign(exposed=(te.returns_365 > 0).astype(int), stratum=pd.qcut(te.freq_365.rank(method="first"), 4, labels=False))
num_, den_, strata = 0.0, 0.0, []
for s_, d in lk.groupby("stratum"):
    a1, n1 = d.loc[d.exposed == 1, "bought_next_90d"].sum(), (d.exposed == 1).sum()
    a0, n0 = d.loc[d.exposed == 0, "bought_next_90d"].sum(), (d.exposed == 0).sum()
    if n1 and n0:
        N = n1 + n0; num_ += a1 * n0 / N; den_ += a0 * n1 / N
        strata.append(dict(stratum=int(s_), n_wrong=int(n1), rate_wrong=float(a1 / n1), n_fine=int(n0), rate_fine=float(a0 / n0)))
rr = num_ / den_
boot = []
rng_l = np.random.default_rng(SEED)
for _ in range(300):
    smp = lk.iloc[rng_l.integers(0, len(lk), len(lk))]; nu = de = 0.0
    for s_, d in smp.groupby("stratum"):
        a1, n1 = d.loc[d.exposed == 1, "bought_next_90d"].sum(), (d.exposed == 1).sum()
        a0, n0 = d.loc[d.exposed == 0, "bought_next_90d"].sum(), (d.exposed == 0).sum()
        if n1 and n0:
            N = n1 + n0; nu += a1 * n0 / N; de += a0 * n1 / N
    boot.append(nu / de)
import json as _json
link = {"definition": "customers with a returned/cancelled order in the last 365 days vs none, compared at similar activity levels",
        "outcome": "buys again within 90 days", "rr": float(rr), "rr_ci": [float(x) for x in np.percentile(boot, [2.5, 97.5])],
        "rate_wrong": float(lk.loc[lk.exposed == 1, "bought_next_90d"].mean()), "n_wrong": int(lk.exposed.sum()),
        "rate_fine": float(lk.loc[lk.exposed == 0, "bought_next_90d"].mean()), "n_fine": int((lk.exposed == 0).sum()), "strata": strata}
(OUT_DIR / "retail_link.json").write_text(_json.dumps(link, indent=1))
print(f"Link (activity-matched risk ratio): {rr:.2f} ({link['rr_ci'][0]:.2f}-{link['rr_ci'][1]:.2f}); saved retail_link.json and retail_orders_scored.csv")

latest = customer_features(END + pd.Timedelta(seconds=1)).drop(columns=["bought_next_90d"])
latest["p_buy_next_90d"] = final_r.predict_proba(latest[R_FEATS])[:, 1]
latest.to_csv(OUT_DIR / "retail_customers_scored.csv", index=False)
print(f"Saved retail_retention_model.joblib, retail_order_risk_model.joblib, retail_orders_scored.csv, retail_link.json, retail_customers_scored.csv ({len(latest):,} active customers at {END.date()})")

## Summary (computed by this run)

In [ ]:
print(f"R1 retention ({R_BEST}): AUC {rb['auc']:.3f} ({rb['ci'][0]:.3f}-{rb['ci'][1]:.3f}) vs best simple rule {best_rule} "
      f"{base_rules[best_rule]['auc']:.3f} -> {'PASS' if R_PASS else 'FAIL'}. Olist v2.1 for comparison: 0.557.")
print(f"O1 order risk ({O_BEST}): AUC {ob['auc']:.3f} ({ob['ci'][0]:.3f}-{ob['ci'][1]:.3f}) vs best rule {max(o_rules.values()):.3f} "
      f"-> {'PASS' if O_PASS else 'FAIL'}. Olist v3 for comparison: 0.628.")
print("L1: see the activity-matched table - only a gap that holds WITHIN activity levels suggests returns go with lower retention.")